# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset contains survey data on mental health indicators among residents of Kilifi County, with demographic details and measurements from GAD-7, PHQ-9, and PSQ assessments.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview

Review available record sets and fields by their Croissant `@id`s for reference in further code. This is important for dynamic data access and reproducibility.

In [ ]:
# List all record sets and their fields with @id references
record_sets = list(dataset.record_sets)
print(f"Record Sets available: {len(record_sets)}\n")
for rset in record_sets:
    print(f"Record Set Name: {rset.name}\n  @id: {rset.id}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    print()


Let's view the first few records from each record set using the `@id` identifiers as references. Common record set `@id`s found in this dataset are similar to `'cr:RecordSet_Answers'` and `'cr:RecordSet_Demographics'` -- adjust as found above.

In [ ]:
# Show very first record for each record set, referenced by @id
for rset in record_sets:
    print(f"Sample records from Record Set: {rset.name} (id: {rset.id})")
    gen = dataset.records(record_set=rset.id)
    for i, rec in enumerate(gen):
        print(rec)
        if i >= 0:  # Show only the first record
            break
    print('-' * 40)

## 3. Data Extraction

Load the main tabular record set(s) into pandas DataFrame(s) using their record set `@id`s for analysis. The Croissant schema typically defines at least one main record set with survey responses.

In [ ]:
# Gather all tabular record set @id's found above
main_recordset_ids = [rs.id for rs in record_sets]

# We'll load each record set into a DataFrame
dataframes = {}
for rs_id in main_recordset_ids:
    print(f"Loading record set: {rs_id}")
    data = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(data)
    dataframes[rs_id] = df
    # Show just some sample columns and rows
    print(df.columns.tolist())
    print(df.head(2))
    print('-' * 30)


## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field for filtering and normalization, and a categorical field for grouping, based on what is present. This demonstration uses the correct record set and field `@id`s found above. Adjust these IDs to your exact dataset schema if they differ.

In [ ]:
# Set IDs for main record set and fields, change as necessary based on cell 2 results:
# Example:
# record_set_id = 'cr:RecordSet_MainSurvey'  # Replace with correct @id
# numeric_field_id = 'gad7_total'  # Or '@id' for GAD-7 total score
# group_field_id = 'gender'  # Or '@id' for gender

# For demonstration, let's try to auto-detect sensible fields
# We'll use the first record set and try to select a numeric field
record_set_id = main_recordset_ids[0]
df = dataframes[record_set_id]

# Find numeric-like fields (look for int/float columns)
numeric_candidates = df.select_dtypes(include=['int', 'float']).columns.tolist()
if not numeric_candidates and len(df) > 0:
    # Fallback: look for columns with likely score names
    numeric_candidates = [c for c in df.columns if 'score' in c.lower() or 'total' in c.lower()]

print("Available numeric fields:", numeric_candidates)
numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]

# Find a group field (categorical)
group_candidates = [c for c in df.columns if any(key in c.lower() for key in ['gender','sex','age','education','village','marital'])]
group_field = group_candidates[0] if group_candidates else df.columns[0]

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Filtering records: e.g. keep only those with score > threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (show up to 5 rows):")
print(filtered_df.head())

# Add a normalized score column (z-score)
normed_col = f"{numeric_field}_normalized"
filtered_df[normed_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records (showing both columns):")
print(filtered_df[[numeric_field, normed_col]].head())

# Group by categorical field, show mean of numeric field
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean normalized {numeric_field} grouped by {group_field}:")
    print(grouped_df)
else:
    print(f"Column {group_field} not found for grouping.")

## 5. Visualization

Let's plot a simple distribution of a numeric score (e.g., GAD-7, PHQ-9, or PSQ, as available) and see the mean by demographic group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution Plot
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Grouped bar plot (mean score by group)
if group_field in df.columns:
    plt.figure(figsize=(8,4))
    means = df.groupby(group_field)[numeric_field].mean().reset_index()
    sns.barplot(x=group_field, y=numeric_field, data=means, palette='turbo')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We:
- Inspected dataset metadata, record sets, and schema via standard Croissant `@id` references.
- Loaded tabular survey data into pandas DataFrames.
- Filtered and normalized numeric assessment scores (e.g., GAD-7, PHQ-9) and grouped results by demographic fields.
- Visualized score distributions and means by demographic groups.

This approach can be adapted for other Croissant-described datasets. For further analysis, consult the [dataset documentation](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) and refer to the schema's field `@id`s for reproducible workflows.